In [0]:
DROP SCHEMA IF EXISTS gold_olist CASCADE;
CREATE SCHEMA IF NOT EXISTS gold_olist;


In [0]:
USE SCHEMA silver_olist

In [0]:
SHOW TABLES

#####1.Modify & load dimension tables to gold_layer: customers, products, sellers,payment,reviews

1.1. dim_customer

In [0]:
select * from customers limit 1

In [0]:
CREATE OR REPLACE TABLE gold_olist.dim_customer AS
SELECT
  customer_unique_id,
  zip_code_prefix,
  city,
  state
FROM 
  customers


1.2. dim_seller

In [0]:
--View sellers table
SELECT
 *
FROM 
  sellers
LIMIT 5

In [0]:
-- Load sellers table to gold layer
CREATE OR REPLACE TABLE gold_olist.dim_seller AS
  SELECT
 *
FROM 
  sellers

1.3 dim_product

In [0]:
--View table products
SELECT * FROM products limit 5

In [0]:
--Load to gold layer
CREATE OR REPLACE TABLE gold_olist.dim_product AS
SELECT *
FROM products

1.4. Dim_payment

In [0]:
--View payment table
SELECT * FROM order_payments limit 5

In [0]:
CREATE OR REPLACE TABLE gold_olist.dim_payment AS
  SELECT * FROM order_payments

1.5. Dim_review

In [0]:
--View order_payments table
SELECT * FROM order_reviews limit 5

In [0]:
CREATE OR REPLACE TABLE gold_olist.dim_review AS
SELECT 
  order_id,
  review_id, 
  review_score
FROM 
  order_reviews

#####2. Fact_orders & Fact_order_item

2.1. Fact_order_item

In [0]:
--View order_items table
SELECT * FROM order_items limit 5

In [0]:
--Load order_items to gold layer
CREATE OR REPLACE TABLE gold_olist.fact_order_item AS 
SELECT * FROM order_items

2.1. Fact_orders

In [0]:
--Calculate each order value
SELECT * FROM order_items limit 5

In [0]:
CREATE OR REPLACE TABLE gold_olist.fact_order AS
WITH order_details AS --calculate details of each order such as total_items, goods_values, shipping_fee, and payment_amount
  (SELECT order_id, 
  COUNT(product_id) as total_items,
  ROUND(SUM(price),2) as goods_value, 
  ROUND(SUM(freight_value),2) as shipping_fee,
  ROUND(SUM(freight_value+price),2) as payment_amount
  FROM order_items
  GROUP BY order_id),
payment_details AS(--calculate paid value of each order 
  SELECT order_id,
  ROUND(SUM(payment_value),2) as paid
  FROM order_payments
  GROUP BY order_id
),
new_order_id AS--Left join orders with order_details cte, payment_details ctes, and customers. 
  (SELECT 
      o.order_id,
      c.customer_unique_id,
      o.order_status,
      o.order_purchase_ts,
      o.order_purchase_date,
      o.order_approved_ts,
      o.order_approved_date,
      o.order_delivered_carrier_ts,
      o.order_delivered_carrier_date,
      o.order_delivered_customer_ts,
      o.order_delivered_customer_date,
      o.order_estimated_delivery_date,
      COALESCE(od.total_items,0) AS total_items,
      COALESCE(od.goods_value,0) AS goods_values,
      COALESCE(od.shipping_fee,0) AS shipping_fee,
      COALESCE(od.payment_amount,0) AS payment_amount,
      COALESCE(pd.paid,0) AS paid
  FROM orders o
  LEFT JOIN order_details od
  ON o.order_id = od.order_id
  LEFT JOIN payment_details pd
  ON o.order_id = pd.order_id
  LEFT JOIN customers c
  ON o.customer_id = c.customer_id
  )
--Remove invaid rows that having paid amount more than payment_amount of each order, it is possible that customer has paid 1 cent more than payment_amount. 
  SELECT * FROM new_order_id 
  WHERE paid<= payment_amount+0.1

#####3. Download all tables in gold layers to csv

3.1. Copy tables to temp dbfs

In [0]:
%python
#Set Schema name and output dbfs folder
schema_name = "gold_olist"
output_dbfs_base = "dbfs:/tmp/gold_olist_tables"

# clean old folder if exists
dbutils.fs.rm(output_dbfs_base, recurse=True)
dbutils.fs.mkdirs(output_dbfs_base)

print("DBFS folder ready:", output_dbfs_base)



In [0]:
%python
#List all tables in gold_schema: 
tables = [t.name for t in spark.catalog.listTables(schema_name)]
print(f"Tables found in {schema_name}:", tables)


In [0]:
%python
#Loop through all tables and write to dbfs
for table in tables:
    print("Exporting:", table)
    
    # Load the table
    df = spark.table(f"{schema_name}.{table}")
    
    # Set DBFS output folder for this table
    table_dbfs_dir = f"{output_dbfs_base}/{table}"
    
    # Save as CSV (one file with header)
    df.coalesce(1).write.mode("overwrite").option("header", "true").csv(table_dbfs_dir)
    
    csv_part = [f.path for f in dbutils.fs.ls(table_dbfs_dir) if f.name.endswith(".csv")][0]
    dbutils.fs.cp(csv_part, f"dbfs:/FileStore/gold_olist_tables/{table}.csv")
    
    print(f"{schema_name}.{table} saved --> dbfs:/FileStore/gold_olist_tables/{table}.csv")


In [0]:
%python
# Check dbfs csv folder in FileStore folder: 
display(dbutils.fs.ls('dbfs:/FileStore/gold_olist_tables/'))



3.2. Download csv files

In [0]:
%python
import shutil
import os

# local driver temp folder
tmp_local = "/tmp/gold_olist_csvs_zip"
os.makedirs(tmp_local, exist_ok=True)

# copy all CSVs from DBFS to local driver
for table in tables:
    dbutils.fs.cp(f"dbfs:/FileStore/gold_olist_tables/{table}.csv",
                  f"file:{tmp_local}/{table}.csv")

# zip them
shutil.make_archive("/tmp/gold_olist_tables", 'zip', tmp_local)

# copy zip to FileStore
dbutils.fs.cp("file:/tmp/gold_olist_tables.zip", "dbfs:/FileStore/gold_olist_tables.zip")


Then, the zip file is downloaded in this link: https://<workspace>.azuredatabricks.net/files/gold_olist_tables.zip